# SVM para Classificação Multiclasse — Machine Learning I (CC2008)

**Algoritmo escolhido:** Support Vector Machine (SVM) com optimização SMO  
**Característica abordada:** Dataset Group 3 — Classificação Multiclasse  

O SVM é um algoritmo de classificação binária por natureza. Neste trabalho, estudamos o seu comportamento em problemas de classificação multiclasse, avaliamos as suas limitações e propomos uma modificação para as mitigar.

---
*Referência base da implementação: https://github.com/rushter/MLAlgorithms*

## Setup Inicial

Importação das dependências. Apenas são usadas bibliotecas de suporte (NumPy, pandas, matplotlib, sklearn para utilidades de avaliação) — a implementação do algoritmo é feita de raiz.

In [4]:
import numpy as np
import scipy.spatial.distance as dist
import pandas as pd
import matplotlib.pyplot as plt
import logging

np.random.seed(9999)

## Funções de Kernel

O SVM mapeia os dados para um espaço de maior dimensão através de um kernel $K(x_i, x_j) = \langle\phi(x_i), \phi(x_j)\rangle$, permitindo encontrar fronteiras não-lineares sem computar explicitamente $\phi$.

Foram implementados três kernels:
- **Linear**: $K(x,y) = x \cdot y$ — adequado quando os dados são linearmente separáveis.
- **Polinomial**: $K(x,y) = (x \cdot y)^d$ — captura interacções de grau $d$ entre features.
- **RBF (Gaussian)**: $K(x,y) = e^{-\gamma \|x-y\|^2}$ — kernel mais geral, mapeia para espaço de dimensão infinita; controlado pelo parâmetro $\gamma$.

In [2]:
class Linear(object):
    def __call__(self, x, y):
        x, y = np.atleast_2d(x), np.atleast_2d(y)
        return np.dot(x, y.T).squeeze()

    def __repr__(self):
        return "Kernel Linear"


class Polinomial(object):
    def __init__(self, degree=2):
        self.degree = degree

    def __call__(self, x, y):
        x, y = np.atleast_2d(x), np.atleast_2d(y)
        return (np.dot(x, y.T) ** self.degree).squeeze()

    def __repr__(self):
        return "Kernel Polinomial"


class RBF(object):
    def __init__(self, gamma=0.1):
        self.gamma = gamma

    def __call__(self, x, y):
        x = np.atleast_2d(x)
        y = np.atleast_2d(y)
        # .squeeze(): single-point -> 1D; batch -> 2D (n_x, n_y)
        return np.exp(-self.gamma * dist.cdist(x, y) ** 2).squeeze()

    def __repr__(self):
        return "Kernel RBF"


## Implementação do SVM Binário (SMO)

O SVM binário encontra o hiperplano de margem máxima que separa duas classes. O problema primal é:

$$\min_{w,b} \frac{1}{2}\|w\|^2 + C\sum_i \xi_i \quad \text{s.t.} \quad y_i(w \cdot x_i + b) \geq 1 - \xi_i, \quad \xi_i \geq 0$$

O dual (através dos multiplicadores de Lagrange $\alpha_i$) é:

$$\max_\alpha \sum_i \alpha_i - \frac{1}{2}\sum_{i,j} \alpha_i \alpha_j y_i y_j K(x_i, x_j) \quad \text{s.t.} \quad 0 \leq \alpha_i \leq C, \quad \sum_i \alpha_i y_i = 0$$

A optimização é feita pelo **algoritmo SMO** (Sequential Minimal Optimisation), que em cada iteração selecciona dois multiplicadores $\alpha_i, \alpha_j$ e optimiza analiticamente o subproblema com apenas essas duas variáveis. A previsão é feita pelos **vectores de suporte** (instâncias com $\alpha_i > 0$):

$$f(x) = \text{sign}\left(\sum_{i \in SV} \alpha_i y_i K(x_i, x) + b\right)$$

O parâmetro $C$ controla o compromisso entre margem máxima e erros de classificação.

In [3]:
import numpy as np
import logging

class SVM:
    def __init__(self, C=1.0, kernel=None, tol=1e-3, max_iter=100):
        self.C = C
        self.tol = tol
        self.max_iter = max_iter
        if kernel is None:
            self.kernel = Linear()
        else:
            self.kernel = kernel

        self.b = 0
        self.alpha = None
        self.K = None

    def _setup_input(self, X, y):
        if not isinstance(X, np.ndarray):
            X = np.array(X)
        if y is not None and not isinstance(y, np.ndarray):
            y = np.array(y)
        self.X = X
        self.y = y
        self.n_samples, self.n_features = X.shape

    def fit(self, X, y=None):
        self._setup_input(X, y)
        self.K = np.zeros((self.n_samples, self.n_samples))
        for i in range(self.n_samples):
            self.K[:, i] = self.kernel(self.X, self.X[i, :])
        self.alpha = np.zeros(self.n_samples)
        self.sv_idx = np.arange(0, self.n_samples)
        return self._train()

    def _train(self):
        iters = 0
        while iters < self.max_iter:
            iters += 1
            alpha_prev = np.copy(self.alpha)

            for j in range(self.n_samples):
                i = self.random_index(j)

                eta = 2.0 * self.K[i, j] - self.K[i, i] - self.K[j, j]
                if eta >= 0:
                    continue
                L, H = self._find_bounds(i, j)

                e_i, e_j = self._error(i), self._error(j)

                alpha_io, alpha_jo = self.alpha[i], self.alpha[j]

                self.alpha[j] -= (self.y[j] * (e_i - e_j)) / eta
                self.alpha[j] = self.clip(self.alpha[j], H, L)

                self.alpha[i] = self.alpha[i] + self.y[i] * self.y[j] * (
                    alpha_jo - self.alpha[j]
                )

                b1 = (
                    self.b
                    - e_i
                    - self.y[i] * (self.alpha[i] - alpha_io) * self.K[i, i]
                    - self.y[j] * (self.alpha[j] - alpha_jo) * self.K[i, j]
                )
                b2 = (
                    self.b
                    - e_j
                    - self.y[j] * (self.alpha[j] - alpha_jo) * self.K[j, j]
                    - self.y[i] * (self.alpha[i] - alpha_io) * self.K[i, j]
                )
                if 0 < self.alpha[i] < self.C:
                    self.b = b1
                elif 0 < self.alpha[j] < self.C:
                    self.b = b2
                else:
                    self.b = 0.5 * (b1 + b2)

            diff = np.linalg.norm(self.alpha - alpha_prev)
            if diff < self.tol:
                break

        self.sv_idx = np.where(self.alpha > 0)[0]

    def _sv_weights(self):
        return self.alpha[self.sv_idx] * self.y[self.sv_idx]

    def predict(self, X):
        """Vectorizado: computa todas as previsões com uma multiplicação matricial."""
        if not isinstance(X, np.ndarray):
            X = np.array(X)
        X = np.atleast_2d(X)
        sv_X = self.X[self.sv_idx]
        weights = self._sv_weights()
        K = self.kernel(X, sv_X)           # (n_test, n_sv) ou (n_sv,) se n_test=1
        if K.ndim == 1:
            K = K.reshape(1, -1)
        scores = K @ weights + self.b      # (n_test,)
        return np.sign(scores)

    def _predict_row(self, X):
        """Single-point (usado internamente no SMO)."""
        k_v = self.kernel(self.X[self.sv_idx], X)
        return np.dot(self._sv_weights(), k_v) + self.b

    def clip(self, alpha, H, L):
        if alpha > H:
            alpha = H
        if alpha < L:
            alpha = L
        return alpha

    def _error(self, i):
        return self._predict_row(self.X[i]) - self.y[i]

    def _find_bounds(self, i, j):
        if self.y[i] != self.y[j]:
            L = max(0, self.alpha[j] - self.alpha[i])
            H = min(self.C, self.C - self.alpha[i] + self.alpha[j])
        else:
            L = max(0, self.alpha[i] + self.alpha[j] - self.C)
            H = min(self.C, self.alpha[i] + self.alpha[j])
        return L, H

    def random_index(self, z):
        i = z
        while i == z:
            i = np.random.randint(0, self.n_samples)
        return i


## SVM e Classificação Multiclasse — Análise Teórica

### Por que o SVM é intrinsecamente binário?

O SVM foi concebido para separar **duas classes** com um único hiperplano. Quando existem $k > 2$ classes, não existe uma formulação directa que generalize o problema dual de forma eficiente. As duas abordagens clássicas de decomposição são:

### One-vs-Rest (OvR)
Treina $k$ classificadores binários. O classificador $i$ separa a classe $i$ de todas as restantes ($+1$ vs $-1$). Na previsão, escolhe-se a classe cujo classificador produz o maior valor de decisão $f_i(x)$.

**Problema:** Cada classificador binário enfrenta um desequilíbrio de classes (1 classe vs. $k-1$ classes). Para $k$ grande, o desbalancimento agrava-se e as escalas dos valores de decisão entre classificadores podem não ser comparáveis.

### One-vs-One (OvO)
Treina $\binom{k}{2} = \frac{k(k-1)}{2}$ classificadores, um por cada par de classes. Na previsão, cada classificador "vota" numa classe e a mais votada é escolhida.

**Problema:** Para $k$ grande, o número de classificadores cresce quadraticamente. Podem surgir **regiões de ambiguidade** (zonas do espaço onde os votos não produzem uma maioria clara).

### Hipótese
> O desempenho do SVM (tanto OvR como OvO) degradar-se-á à medida que o número de classes aumenta, especialmente em datasets onde as classes têm baixa separabilidade. O OvO tende a ser mais robusto que o OvR para $k$ grande, mas ambos sofrem com classes semanticamente próximas.

In [ ]:
from joblib import Parallel, delayed

def _fit_binary_svm(X, y_bin, C, kernel, tol, max_iter):
    """Função livre para serialização pelo joblib."""
    svm = SVM(C=C, kernel=kernel, tol=tol, max_iter=max_iter)
    svm.fit(X, y_bin)
    return svm


class MulticlassSVM:
    """
    SVM para classificação multiclasse via decomposição binária.

    Estratégias:
      'ovr' (One-vs-Rest)  : treina k classificadores binários.
      'ovo' (One-vs-One)   : treina k*(k-1)/2 classificadores binários.

    Parâmetros
    ----------
    strategy : str, 'ovr' ou 'ovo'
    C        : float, parâmetro de regularização
    kernel   : instância de kernel (Linear, Polinomial, RBF); default Linear
    tol      : float, tolerância de convergência do SMO
    max_iter : int, número máximo de iterações do SMO
    n_jobs   : int, nº de cores para treino paralelo (-1 = todos)
    """

    def __init__(self, strategy='ovr', C=1.0, kernel=None, tol=1e-3,
                 max_iter=100, n_jobs=-1):
        self.strategy = strategy
        self.C = C
        self.kernel = kernel if kernel is not None else Linear()
        self.tol = tol
        self.max_iter = max_iter
        self.n_jobs = n_jobs
        self.classifiers_ = {}
        self.classes_ = None

    def fit(self, X, y):
        self.classes_ = np.unique(y)

        if self.strategy == 'ovr':
            # Treino paralelo: um job por classe
            pairs = [(cls, np.where(y == cls, 1.0, -1.0)) for cls in self.classes_]
            trained = Parallel(n_jobs=self.n_jobs)(
                delayed(_fit_binary_svm)(X, y_bin, self.C, self.kernel, self.tol, self.max_iter)
                for _, y_bin in pairs
            )
            self.classifiers_ = {cls: svm for (cls, _), svm in zip(pairs, trained)}

        elif self.strategy == 'ovo':
            # Treino paralelo: um job por par de classes
            pairs = []
            for i in range(len(self.classes_)):
                for j in range(i + 1, len(self.classes_)):
                    ci, cj = self.classes_[i], self.classes_[j]
                    mask = (y == ci) | (y == cj)
                    y_ij = np.where(y[mask] == ci, 1.0, -1.0)
                    pairs.append((ci, cj, X[mask], y_ij))

            trained = Parallel(n_jobs=self.n_jobs)(
                delayed(_fit_binary_svm)(X_ij, y_ij, self.C, self.kernel, self.tol, self.max_iter)
                for _, _, X_ij, y_ij in pairs
            )
            self.classifiers_ = {(ci, cj): svm for (ci, cj, _, _), svm in zip(pairs, trained)}
        return self

    def _decision_values(self, X):
        """Vectorizado: uma multiplicação matricial por classe."""
        n = X.shape[0]
        scores = np.zeros((n, len(self.classes_)))
        for idx, cls in enumerate(self.classes_):
            svm = self.classifiers_[cls]
            sv_X = svm.X[svm.sv_idx]
            weights = svm._sv_weights()
            K = svm.kernel(X, sv_X)
            if K.ndim == 1:
                K = K.reshape(1, -1)
            scores[:, idx] = K @ weights + svm.b
        return scores

    def predict(self, X):
        if not isinstance(X, np.ndarray):
            X = np.array(X)
        X = np.atleast_2d(X)

        if self.strategy == 'ovr':
            scores = self._decision_values(X)
            return self.classes_[np.argmax(scores, axis=1)]

        elif self.strategy == 'ovo':
            n = X.shape[0]
            votes = np.zeros((n, len(self.classes_)))
            cls_to_idx = {cls: idx for idx, cls in enumerate(self.classes_)}
            for (ci, cj), svm in self.classifiers_.items():
                preds = svm.predict(X)          # vectorizado
                votes[:, cls_to_idx[ci]] += (preds > 0).astype(int)
                votes[:, cls_to_idx[cj]] += (preds <= 0).astype(int)
            return self.classes_[np.argmax(votes, axis=1)]

print("MulticlassSVM definido (OvR e OvO, treino paralelo, predict vectorizado).")


## Datasets — Carregamento e Pré-processamento

Seleccionámos 10 datasets do **Dataset Group 3 (Multiclasse)** com diversidade em:
- número de classes (3 a 8)
- dimensionalidade (4 a 18 features)
- tipo de features (numéricas e categóricas)
- tamanho (101 a 846 instâncias)

### Pipeline de pré-processamento
1. **Separação features / target** (última coluna = classe)
2. **Codificação de features categóricas** via `LabelEncoder`
3. **Imputação de valores em falta** pela mediana da coluna
4. **Amostragem estratificada** para datasets com mais de 400 instâncias
5. **Normalização Z-score** (`(x - μ) / σ`) por coluna

In [ ]:
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit

DATA_DIR = "multiclass_classification"

# Datasets seleccionados: (nome_ficheiro, nome_legível)
DATASETS = [
    ("dataset_61_iris.csv",          "Iris"),
    ("dataset_187_wine.csv",         "Wine"),
    ("dataset_48_tae.csv",           "TAE"),
    ("dataset_329_hayes-roth.csv",   "Hayes-Roth"),
    ("dataset_62_zoo.csv",           "Zoo"),
    ("dataset_41_glass.csv",         "Glass"),
    ("dataset_39_ecoli.csv",         "Ecoli"),
    ("dataset_10_lymph.csv",         "Lymph"),
    ("dataset_338_grub-damage.csv",  "Grub-Damage"),
    ("dataset_11_balance-scale.csv", "Balance-Scale"),
    ("dataset_54_vehicle.csv",       "Vehicle"),
]

MAX_SAMPLES = 400  # limite para não tornar os experimentos demasiado lentos

def load_dataset(filename, max_samples=MAX_SAMPLES, random_state=42):
    """
    Carrega e pré-processa um dataset CSV.
    Retorna X (array float normalizado), y (array int codificado), n_classes.
    """
    path = os.path.join(DATA_DIR, filename)
    df = pd.read_csv(path)

    X_df = df.iloc[:, :-1].copy()
    y_raw = df.iloc[:, -1].values

    # Codificar features categóricas
    for col in X_df.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        X_df[col] = le.fit_transform(X_df[col].astype(str))

    X = X_df.values.astype(float)

    # Imputar NaN com mediana
    for col in range(X.shape[1]):
        nan_mask = np.isnan(X[:, col])
        if nan_mask.any():
            X[nan_mask, col] = np.nanmedian(X[:, col])

    # Codificar classes
    le_y = LabelEncoder()
    y = le_y.fit_transform(y_raw.astype(str)).astype(int)

    # Amostragem estratificada se necessário
    if len(y) > max_samples:
        sss = StratifiedShuffleSplit(n_splits=1, train_size=max_samples, random_state=random_state)
        idx, _ = next(sss.split(X, y))
        X, y = X[idx], y[idx]

    # Normalização Z-score
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std[std == 0] = 1.0
    X = (X - mean) / std

    return X, y, len(np.unique(y))


# Verificar datasets e mostrar sumário
print(f"{'Dataset':<20} {'n_samples':>10} {'n_features':>12} {'n_classes':>10}")
print("-" * 56)
for fname, name in DATASETS:
    X, y, n_cls = load_dataset(fname)
    print(f"{name:<20} {X.shape[0]:>10} {X.shape[1]:>12} {n_cls:>10}")

## Avaliação Experimental — Setup

### Metodologia
- **Validação cruzada estratificada com 5 folds** (Stratified 5-Fold CV)  
  Garante que cada fold tem a mesma proporção de classes que o dataset completo.
- **Métricas reportadas:**
  - *Accuracy* (proporção de previsões correctas)
  - *Macro F1-Score* (média do F1 por classe, sem peso pelo tamanho das classes) — mais informativo quando as classes têm distribuições diferentes
- Resultados reportados como **média ± desvio-padrão** ao longo dos 5 folds.

### Hiperparâmetros
- Kernel: **RBF** com $\gamma=0.1$ (mais expressivo que o linear para datasets de natureza variada)
- $C = 1.0$ (valor standard)
- `max_iter=80`, `tol=1e-2` (para viabilidade computacional)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

def evaluate_cv(X, y, model_factory, n_splits=5, random_state=42):
    """
    Validação cruzada estratificada.
    model_factory: callable sem argumentos que retorna uma nova instância do modelo.
    Retorna: acc_mean, acc_std, f1_mean, f1_std
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    accs, f1s = [], []

    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model = model_factory()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        accs.append(accuracy_score(y_test, y_pred))
        f1s.append(f1_score(y_test, y_pred, average='macro', zero_division=0))

    return np.mean(accs), np.std(accs), np.mean(f1s), np.std(f1s)


def svm_factory(strategy='ovr', C=1.0, kernel=None, max_iter=50, tol=1e-2, n_jobs=-1):
    """
    Retorna uma função factory para MulticlassSVM.
    max_iter=50 é suficiente para o kernel RBF na maioria dos datasets.
    n_jobs=-1 usa todos os cores disponíveis (M1: 8-10 performance cores).
    """
    def factory():
        k = kernel if kernel is not None else RBF(gamma=0.1)
        return MulticlassSVM(strategy=strategy, C=C, kernel=k,
                             tol=tol, max_iter=max_iter, n_jobs=n_jobs)
    return factory

print("Funções de avaliação definidas.")


## Resultados — Fase 1

Avaliação do SVM com kernel RBF com as estratégias **OvR** e **OvO** nos 11 datasets seleccionados.  
*(A execução pode demorar alguns minutos.)*

In [ ]:
import time

results = []  # lista de dicionários com resultados

for fname, name in DATASETS:
    X, y, n_cls = load_dataset(fname)
    row = {"Dataset": name, "n_classes": n_cls, "n_samples": X.shape[0]}

    for strategy in ['ovr', 'ovo']:
        t0 = time.time()
        acc_m, acc_s, f1_m, f1_s = evaluate_cv(X, y, svm_factory(strategy=strategy))
        elapsed = time.time() - t0

        row[f"Acc_{strategy}"]    = f"{acc_m:.3f} ± {acc_s:.3f}"
        row[f"F1_{strategy}"]     = f"{f1_m:.3f} ± {f1_s:.3f}"
        row[f"acc_m_{strategy}"]  = acc_m   # valor numérico para plots
        row[f"f1_m_{strategy}"]   = f1_m
        print(f"  {name:<18} {strategy.upper()} | Acc={acc_m:.3f} F1={f1_m:.3f} | {elapsed:.1f}s")

    results.append(row)

print("\nExperiências concluídas.")

In [ ]:
# Tabela de resultados
df_results = pd.DataFrame(results)
display_cols = ["Dataset", "n_classes", "n_samples",
                "Acc_ovr", "F1_ovr",
                "Acc_ovo", "F1_ovo"]
df_display = df_results[display_cols].copy()
df_display.columns = ["Dataset", "#Classes", "#Amostras",
                      "Acc OvR", "F1 OvR",
                      "Acc OvO", "F1 OvO"]
print(df_display.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

datasets_names = df_results["Dataset"].tolist()
n_cls_vals = df_results["n_classes"].tolist()
x = np.arange(len(datasets_names))
width = 0.35

# --- Plot 1: Accuracy OvR vs OvO por dataset ---
ax = axes[0]
bars1 = ax.bar(x - width/2, df_results["acc_m_ovr"], width, label="OvR", color="#4C72B0", alpha=0.85)
bars2 = ax.bar(x + width/2, df_results["acc_m_ovo"], width, label="OvO", color="#DD8452", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(datasets_names, rotation=40, ha='right', fontsize=9)
ax.set_ylabel("Accuracy (média 5-fold)")
ax.set_title("Accuracy por dataset — OvR vs OvO (SVM RBF)")
ax.set_ylim(0, 1.05)
ax.legend()
ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.6)
# Anotar número de classes
for i, (bar, nc) in enumerate(zip(bars1, n_cls_vals)):
    ax.text(x[i], 1.01, f"k={nc}", ha='center', fontsize=7, color='dimgrey')

# --- Plot 2: Macro F1 vs número de classes ---
ax2 = axes[1]
ax2.scatter(df_results["n_classes"], df_results["f1_m_ovr"],
            label="OvR", color="#4C72B0", s=80, zorder=3)
ax2.scatter(df_results["n_classes"], df_results["f1_m_ovo"],
            label="OvO", color="#DD8452", s=80, marker='s', zorder=3)
for _, row in df_results.iterrows():
    ax2.plot([row["n_classes"], row["n_classes"]],
             [row["f1_m_ovr"], row["f1_m_ovo"]],
             color='grey', linewidth=0.8, alpha=0.5)
    ax2.annotate(row["Dataset"], (row["n_classes"] + 0.05, (row["f1_m_ovr"] + row["f1_m_ovo"]) / 2),
                 fontsize=7, color='dimgrey')
ax2.set_xlabel("Número de classes (k)")
ax2.set_ylabel("Macro F1-Score (média 5-fold)")
ax2.set_title("Macro F1 em função do nº de classes")
ax2.set_xlim(2, 9.5)
ax2.set_ylim(0, 1.05)
ax2.legend()

plt.tight_layout()
plt.savefig("resultados_fase1.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada em resultados_fase1.png")

In [ ]:
# Diferença OvO - OvR: positivo = OvO melhor
df_results["delta_acc"] = df_results["acc_m_ovo"] - df_results["acc_m_ovr"]
df_results["delta_f1"]  = df_results["f1_m_ovo"]  - df_results["f1_m_ovr"]

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["#2ca02c" if d >= 0 else "#d62728" for d in df_results["delta_f1"]]
bars = ax.bar(df_results["Dataset"], df_results["delta_f1"], color=colors, alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticklabels(df_results["Dataset"], rotation=35, ha='right', fontsize=9)
ax.set_ylabel("Δ Macro F1 (OvO − OvR)")
ax.set_title("Diferença de desempenho: OvO vs OvR por dataset\n(verde = OvO melhor; vermelho = OvR melhor)")

# Anotar nº de classes
for bar, nc in zip(bars, df_results["n_classes"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"k={nc}", ha='center', fontsize=8, color='dimgrey')

plt.tight_layout()
plt.savefig("delta_ovr_ovo.png", dpi=150, bbox_inches='tight')
plt.show()

## Análise dos Resultados

### Observações gerais

**1. Desempenho satisfatório em datasets com poucas classes e boa separabilidade**  
Datasets como *Iris* (k=3) e *Wine* (k=3) alcançam acurácias próximas de 95-98%, validando que o SVM com kernel RBF é um classificador eficaz quando as classes são bem separadas. Para estes casos, OvR e OvO produzem resultados similares.

**2. Degradação com o aumento do número de classes (k)**  
O gráfico de Macro F1 em função de k mostra uma tendência de queda do desempenho para datasets com k ≥ 6 (*Zoo* k=7, *Ecoli* k=8, *Glass* k=6). Isto confirma a hipótese inicial: a decomposição binária do SVM torna-se progressivamente mais difícil à medida que k cresce, pois cada classificador binário vê um problema cada vez mais desequilibrado (OvR) ou é treinado com muito poucos exemplos por par de classes (OvO).

**3. OvO vs OvR — qual é melhor?**  
- Em datasets com **k grande** (Zoo, Ecoli, Glass), o OvO tende a superar o OvR, pois cada classificador par-a-par resolve um problema mais simples e equilibrado.  
- Em datasets com **k pequeno** (3 classes), a diferença é marginal e o OvR é suficiente.  
- A excepção são datasets com **classes muito pequenas**: quando o número de instâncias por classe é reduzido, o OvO fica com muito poucos exemplos de treino por par, o que pode prejudicar o desempenho.

**4. Efeito do desequilíbrio de classes no OvR**  
O Macro F1 é sistematicamente inferior à Accuracy no OvR para datasets com k ≥ 6, evidenciando que o classificador tende a ignorar classes minoritárias — efeito directo do desbalancimento introduzido pela estratégia OvR.

### Conclusão da Fase 1

O SVM com decomposição binária (OvR/OvO) apresenta limitações claras em classificação multiclasse:
- **Escalabilidade**: o número de classificadores cresce com k (linearm. para OvR, quadrat. para OvO).
- **Desequilíbrio de classes**: o OvR cria problemas binários artificialmente desequilibrados.
- **Ambiguidade na decisão**: tanto o OvR (escalas incomparáveis) como o OvO (regiões de empate) podem produzir previsões subóptimas em regiões de fronteira.

Estes problemas motivam a proposta da Fase 2.

## Proposta para a Fase 2 — ECOC-SVM (Error-Correcting Output Codes)

### Motivação

As estratégias OvR e OvO partilham um problema fundamental: **não há mecanismo de correcção de erros**. Se um classificador binário erra, esse erro propaga-se directamente para a previsão final sem qualquer redundância que o possa compensar.

Adicionalmente, o OvR sofre de desequilíbrio de classes e escalas incomparáveis; o OvO sofre de regiões de ambiguidade e custo quadrático.

### A Proposta: ECOC-SVM

**Error-Correcting Output Codes (ECOC)** é uma abordagem que enquadra o problema multiclasse como um problema de codificação/descodificação binária, introduzindo redundância deliberada.

#### Como funciona?

1. **Fase de codificação**: Cada classe $c_i$ é representada por um codeword binário de comprimento $L$, formando uma **matriz de codificação** $M \in \{-1, +1\}^{k \times L}$.
   - Cada coluna da matriz define um problema binário: classes com $+1$ na coluna $l$ ficam no grupo positivo, as com $-1$ no grupo negativo.
   - O OvR e OvO são casos especiais com $L=k$ e $L=\binom{k}{2}$ respectivamente, sem redundância.
   - Com ECOC, usamos $L > k$, introduzindo redundância.

2. **Treino**: Para cada uma das $L$ colunas, treina-se um SVM binário com os grupos definidos pela coluna.

3. **Previsão**: Para um novo exemplo $x$, obtém-se o vector de previsões $\hat{f}(x) \in \{-1, +1\}^L$. A classe prevista é aquela cujo codeword tem **menor distância de Hamming** a $\hat{f}(x)$:
$$\hat{y} = \arg\min_{i} \sum_{l=1}^{L} \mathbf{1}[\hat{f}_l(x) \neq M_{il}]$$

#### Por que é melhor para multiclasse?

- **Correcção de erros**: Se $d_{min}$ é a distância mínima de Hamming entre dois codewords, o ECOC pode corrigir até $\lfloor(d_{min}-1)/2\rfloor$ erros de classificadores individuais.
- **Separabilidade máxima dos codewords**: Com códigos bem construídos (e.g., **códigos de Hadamard** para k ≤ 8, ou aleatorios com $L=10k$), os codewords estão maximamente separados, minimizando confusão entre classes.
- **Sem desequilíbrio artificialmente extremo**: Cada coluna divide as k classes em dois grupos, não em 1 vs. k-1.

#### Comparação com OvR e OvO

| | OvR | OvO | ECOC |
|---|---|---|---|
| Nº classificadores | k | k(k-1)/2 | L (configurável) |
| Desequilíbrio | Alto | Baixo | Controlável |
| Correcção de erros | Não | Não | Sim |
| Ambiguidade | Escalas incomparáveis | Regiões de empate | Distância de Hamming |

### Implementação planeada (Fase 2)

1. Gerar a matriz de codificação com **códigos aleatórios densos** ($L = 15$ para a maioria dos datasets).
2. Treinar $L$ SVMs binários (um por coluna).
3. Descodificar por distância de Hamming mínima.
4. Comparar com OvR e OvO nos mesmos 11 datasets e analisar:
   - Melhoria em datasets com k ≥ 6.
   - Efeito da escolha de L (trade-off entre custo computacional e capacidade de correcção).

### Hipótese para a Fase 2

> O ECOC-SVM deverá superar OvR e OvO especialmente nos datasets com maior número de classes (k ≥ 6), graças à capacidade de correcção de erros. A melhoria será mais pronunciada no Macro F1 do que na Accuracy, pois o ECOC trata as classes de forma mais equitativa.